# Mattis EDA

Initial loading and first overview inspection for data in `data/raw/Alternative Medien/`.


In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Set Path
PROJECT_ROOT = Path.cwd().parent
BASE_DIR = PROJECT_ROOT / "data" / "raw" / "Alternative Medien"

print("cwd:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_DIR:", BASE_DIR)
print("exists:", BASE_DIR.exists())

if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR not found: {BASE_DIR}")


cwd: /Users/MattisHaumann/Dev/Thesis/Initial EDA
PROJECT_ROOT: /Users/MattisHaumann/Dev/Thesis
BASE_DIR: /Users/MattisHaumann/Dev/Thesis/data/raw/Alternative Medien
exists: True


In [3]:
# CSV reader (for every source except RT)
def read_csv_resilient(csv_path: Path) -> pd.DataFrame:
    encodings = ("utf-8", "utf-8-sig", "latin-1")

    # normal read
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    # delimiter sniff (python engine)
    for enc in encodings:
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                sep=None,
                engine="python",
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    raise ValueError(f"Could not read: {csv_path}")

In [4]:
dfs_by_source = {}
overview_rows = []

source_dirs = sorted(
    [p for p in BASE_DIR.iterdir() if p.is_dir()],
    key=lambda p: p.name.lower()
)

for source_dir in source_dirs:
    csv_files = sorted(source_dir.rglob("*.csv"))
    frames = []
    failed = 0

    for csv_file in csv_files:
        try:
            part = read_csv_resilient(csv_file)
            part["source"] = source_dir.name
            part["source_file"] = csv_file.name  # keep it simple
            frames.append(part)
        except Exception as e:
            failed += 1
            print(f"[WARN] {source_dir.name} -> {csv_file.name}: {e}")

    df_source = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    dfs_by_source[source_dir.name] = df_source

    overview_rows.append({
        "source": source_dir.name,
        "files_found": len(csv_files),
        "files_failed": failed,
        "total_articles": len(df_source),
        "n_columns": df_source.shape[1],
    })

# RT_de.xlsx at the root
rt_path = BASE_DIR / "RT_de.xlsx"
if rt_path.exists():
    df_rt = pd.read_excel(rt_path)
    df_rt["source"] = "RT_de"
    df_rt["source_file"] = rt_path.name
    dfs_by_source["RT_de"] = df_rt

    overview_rows.append({
        "source": "RT_de",
        "files_found": 1,
        "files_failed": 0,
        "total_articles": len(df_rt),
        "n_columns": df_rt.shape[1],
    })

overview_df = (
    pd.DataFrame(overview_rows)
    .sort_values(["total_articles", "source"], ascending=[False, True])
    .reset_index(drop=True)
)

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns
0,deusch_pravda,496,0,397944,9
1,RT_de,1,0,12390,9
2,Nius_Rohdaten_neu,285,0,4885,29
3,apollo,165,0,3550,7
4,Tichy's Einblick,195,0,3126,8
5,Compact,214,0,2045,6
6,Antispiegel,307,0,914,9
7,Reitschuster,238,0,685,6


In [5]:
src = "deusch_pravda"

if src in dfs_by_source:
    df = dfs_by_source[src]

    if "time" in df.columns:
        # Parse German datetime and drop time part
        parsed = pd.to_datetime(
            df["time"],
            format="%d.%m.%Y, %H:%M",
            errors="coerce"
        )

        df["date"] = parsed.dt.normalize()  # keeps only YYYY-MM-DD

        dfs_by_source[src] = df

        print(f"[OK] {src}: created clean 'date' column (without time)")
    else:
        print(f"[WARN] {src}: no 'time' column found")


[OK] deusch_pravda: created clean 'date' column (without time)


In [6]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

date_rows = []

for source, df in dfs_by_source.items():
    date_min = pd.NaT
    date_max = pd.NaT

    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                date_min = s.min()
                date_max = s.max()
                break

    date_rows.append({
        "source": source,
        "date_min": date_min,
        "date_max": date_max,
    })

date_df = pd.DataFrame(date_rows)

overview_df = overview_df.drop(columns=["date_min", "date_max"], errors="ignore")
overview_df = overview_df.merge(date_df, on="source", how="left")

display(overview_df)


,source,files_found,files_failed,total_articles,n_columns,date_min,date_max
0,deusch_pravda,496,0,397944,9,2025-05-05,2025-09-02
1,RT_de,1,0,12390,9,2024-11-14,2026-01-19
2,Nius_Rohdaten_neu,285,0,4885,29,2025-05-01,2026-02-10
3,apollo,165,0,3550,7,2025-08-12,2026-02-10
4,Tichy's Einblick,195,0,3126,8,2025-08-01,2026-02-11
5,Compact,214,0,2045,6,2025-06-11,2026-02-10
6,Antispiegel,307,0,914,9,2025-04-10,2026-02-10
7,Reitschuster,238,0,685,6,2025-06-10,2026-02-10


# Articles per month

In [10]:
DATE_CANDIDATES = ["Date", "date", "day", "published", "published_at", "datetime", "timestamp"]

monthly_rows = []

for source, df in dfs_by_source.items():
    used_series = None

    # find a usable date column
    for col in DATE_CANDIDATES:
        if col in df.columns:
            s = pd.to_datetime(df[col], errors="coerce").dt.normalize()
            if s.notna().any():
                used_series = s
                break

    if used_series is None:
        continue

    # convert to year-month period
    ym = used_series.dt.to_period("M").astype(str)

    # count articles per month
    counts = ym.value_counts()

    for month, count in counts.items():
        monthly_rows.append({"source": source, "year_month": month, "articles": count})

monthly_df = pd.DataFrame(monthly_rows)

pivot = (
    monthly_df
    .pivot_table(index="source", columns="year_month", values="articles", aggfunc="sum", fill_value=0)
    .sort_index()
)

# optional: sort columns chronologically (YYYY-MM already sorts correctly as strings)
pivot = pivot.reindex(sorted(pivot.columns), axis=1)

display(pivot)


year_month,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02
source,,,,,,,,,,,,,,,,
Antispiegel,0,0,0,0,0,85,118,34,32,92,122,113,85,114,87,32
Compact,0,0,0,0,0,0,0,168,176,207,275,301,280,258,293,87
Nius_Rohdaten_neu,0,0,0,0,0,0,482,447,495,466,563,589,604,509,548,182
RT_de,522,1031,958,939,941,862,895,785,887,687,856,855,898,868,406,0
Reitschuster,0,0,0,0,0,0,0,65,106,72,100,0,88,0,67,22
Tichy's Einblick,0,0,0,0,0,0,0,0,0,458,486,510,483,504,505,180
apollo,0,0,0,0,0,0,0,0,0,176,565,647,643,635,646,238
deusch_pravda,0,0,0,0,0,0,22016,30688,39674,36574,2249,0,0,0,0,0


## Erste Insights
Klare Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Zahlen, aber nur in wenigen Monaten (Mai–September 2025). Sind wahrscheinlich Telegramdaten und nicht klassisch news

- **RT_de** veröffentlicht konstant über einen langen Zeitraum (Ende 2024–Anfang 2026). Sehr strukturiert und institutionell

- **Nius, Apollo, Compact und Tichys Einblick** zeigen relativ regelmäßige monatliche Output-Zahlen. Auch stabile redaktionelle Routinen vermutlich

- **Antispiegel und Reitschuster** haben deutlich geringeres Volumen. Vermutlich kleinere Strukturen oder stärker meinungsgetriebene Formate.

**Fazit (erste EDA):**  
Schon auf dieser deskriptiven Ebene sieht man, dass die untersuchten „alternativen Medien“ sehr unterschiedlich arbeiten – sowohl im Volumen als auch in der zeitlichen Dynamik.


# Length of Articles

In [15]:
# map using lowercase source names to avoid case issues
TEXT_COLUMN_MAP = {
    "antispiegel": "Full_Text",
    "apollo": "Inhalt",
    "compact": "Inhalt",
    "deusch_pravda": "full_text",
    "reitschuster": "Inhalt",
    "nius_rohdaten_neu": "article_text",
    "rt_de": "Full_Text",
    "tichy's einblick": "article_text",
}

length_rows = []

for source, df in dfs_by_source.items():
    key = source.lower()  # normalize
    text_col = TEXT_COLUMN_MAP.get(key)

    if text_col is None or text_col not in df.columns:
        print(f"[WARN] {source}: text column not found or not mapped. (mapped={text_col})")
        continue

    words = df[text_col].astype(str).str.split().str.len()

    length_rows.append({
        "source": source,
        "articles": len(df),
        "text_col": text_col,
        "mean_words": words.mean(),
        "median_words": words.median(),
        "min_words": words.min(),
        "max_words": words.max(),
    })

length_df = (
    pd.DataFrame(length_rows)
    .sort_values("mean_words", ascending=False)
    .reset_index(drop=True)
)

display(length_df)

,source,articles,text_col,mean_words,median_words,min_words,max_words
0,Antispiegel,914,Full_Text,2203.440919,1720.0,170.0,18894.0
1,Reitschuster,685,Inhalt,1584.382482,1466.0,837.0,5014.0
2,Tichy's Einblick,3126,article_text,827.087332,765.0,42.0,5857.0
3,RT_de,12390,Full_Text,629.242696,470.5,7.0,4689.0
4,Nius_Rohdaten_neu,4885,article_text,586.183498,426.0,6.0,64721.0
5,Compact,2045,Inhalt,504.015159,417.0,23.0,4324.0
6,apollo,3550,Inhalt,444.925634,384.0,23.0,2662.0
7,deusch_pravda,397944,full_text,162.234063,114.0,1.0,7820.0


## Erste Insights 

Klare strukturelle Unterschiede zwischen den Quellen.

- **deusch_pravda** hat extrem hohe Volumina in kurzer Zeit. Gleichzeitig ist die durchschnittliche Textlänge deutlich niedriger. Das spricht für aggregierte Inhalte (z. B. Telegram) statt klassischer redaktioneller Langform.

- **RT_de** sowie **Antispiegel** und **Reitschuster** weisen im Schnitt deutlich längere Texte auf. Das deutet eher auf ausführlichere Beiträge oder kommentierende Formate hin.

- **Nius, Apollo, Compact und Tichys Einblick** liegen im mittleren Bereich. Hier sieht man eher typische Online-Artikel-Längen.

Wichtig:  
Die Textdaten sind teilweise noch stark „roh“. Es finden sich z. B. Werbeeinblendungen, wiederholte Textpassagen oder technische Artefakte. Vor tiefergehenden inhaltlichen Analysen (z. B. Sentiment, Topic Modeling) ist daher eine systematische Textbereinigung notwendig.

**Zwischenfazit:**  
Schon in dieser frühen EDA zeigen sich klare Unterschiede in Produktionslogik, Umfang und Struktur der Inhalte. Die Datenqualität und Textbereinigung werden dabei ein zentraler methodischer Schritt für die weitere Analyse sein.


## Dateninspektion pro Quelle

In [16]:
import random

SAMPLE_SIZE = 5

for source, df in dfs_by_source.items():
    print("\n" + "="*80)
    print(f"Source: {source}")
    print("="*80)

    if len(df) == 0:
        print("No data available.")
        continue

    sample_df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42)

    for i, row in sample_df.iterrows():
        print("\n--- Article ---")

        # Title if available
        if "title" in df.columns:
            print("Title:", row.get("title"))
        elif "Title" in df.columns:
            print("Title:", row.get("Title"))

        # Try mapped text column if already defined
        text_col = None
        for col in df.columns:
            if col.lower() in ["full_text", "text", "inhalt", "article_text"]:
                text_col = col
                break

        if text_col:
            text_preview = str(row[text_col])[:1000]
            print("\nText Preview:\n", text_preview)
        else:
            print("No obvious text column found.")



Source: Antispiegel

--- Article ---
Title: Beginnt jetzt die „GPS-Show“?

Text Preview:
 Propaganda
Beginnt jetzt die „GPS-Show“?
Schweden meldet eine große Zunahme der Störungen des GPS in der zivilen Luftfahrt. Schuld ist angeblich natürlich Russland und als Beleg erinnert der Spiegel an die Störung des Fluges von von der Leyen, die sich jedoch als Fake herausgestellt hat. Die westliche Propaganda lügt immer lustiger.
von Anti-Spiegel
6. September 2025 12:00 Uhr
Als am Montag gemeldet wurde, das GPS des Fluges von EU-Kommissionschefin Ursula von der Leyen von Warschau ins bulgarische Plowdiw sei von Russland gestört worden und die Maschine hätte deswegen eine Stunde lang Warteschleifen fliegen müssen, bevor die Piloten landen konnten, war schnell klar, dass die Geschichte frei erfunden war.
Die GPS-Lüge vom Montag
Erstens liegt Plowdiw etwa 200 Kilometer vom Schwarzen Meer entfernt, sodass, wenn Russland das GPS gestört hätte, das GPS im halben Bulgarien hätte gestört sein müssen. 